# EmpowerLens — Kaggle GPU runner

Runs real transformer fine-tunes on Kaggle's free GPU, then makes `results_combined/` downloadable.

**Before running:**
1. Settings → **Accelerator: GPU**, and **Internet: On** (needed for `pip` + model downloads from Hugging Face).
2. The splits in `data/splits_combined/` must already be **committed and pushed** on the branch below — the notebook uses them as-is and never regenerates them.

**This run trains DeBERTa-v3 only** (MentalRoBERTa already succeeded and is commented out
in `MODELS` below) on the `multilabel` task, 3 seeds each, so it lands
side-by-side in the same `results_combined/paper_comparison.csv`:

| tag | HF model id | what it is |
|---|---|---|
| `roberta-base` | `roberta-base` | current baseline (already have numbers for this) |
| `deberta-v3-base` | `microsoft/deberta-v3-base` | general-domain, disentangled attention — best "architecture swap" candidate |
| `mental-roberta-base` | `mental/mental-roberta-base` | RoBERTa continued-pretrained on Reddit mental-health corpora |

Set `MODELS` below to skip `roberta-base` if you already have those checkpoints/results.

This notebook only orchestrates shell commands; all logic lives in `src/`.

In [30]:
import os

# 1. Safely reset the working directory to the Kaggle root
os.chdir('/kaggle/working')

# 2. Wipe the slate clean to prevent duplicate/nested folders
!rm -rf /kaggle/working/*

# 3. Clone the repo freshly from your branch
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "izza-space"
!git clone --branch $BRANCH $REPO_URL empowerlens

# 4. Move into the project directory safely
os.chdir('/kaggle/working/empowerlens')

# 5. Upgrade build tools just in case, then install without the -q flag to see any errors
!pip install --upgrade pip setuptools wheel
!pip install -r requirements-transformer.txt
!pip install sentencepiece protobuf captum

Cloning into 'empowerlens'...
remote: Enumerating objects: 356, done.
remote: Counting objects: 100% (356/356), done.
remote: Compressing objects: 100% (266/266), done.
remote: Total 356 (delta 156), reused 282 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (356/356), 7.91 MiB | 22.00 MiB/s, done.
Resolving deltas: 100% (156/156), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Using cached iterative_stratification-0.1.9-py3-none-any.whl.metadata (1.3 kB)
  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Installing build dependencies ... done
  Gett

In [21]:
# 1a. mental/mental-roberta-base is a GATED model on the Hub — you must (1) accept its
#     terms at https://huggingface.co/mental/mental-roberta-base while logged in, then
#     (2) add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets, top menu) holding a
#     Hugging Face access token with read scope. Without this, training that model 401s.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"[warn] no working HF_TOKEN secret ({e}) — mental/mental-roberta-base will 401 "
          f"until you accept its terms and add the Kaggle secret.")

Logged in to Hugging Face Hub.


In [22]:
# 1b. Show exactly what train.csv is made of before spending GPU hours on it —
#     n_annotated vs n_codipas rows and the per-class breakdown from make_splits_combined.py's manifest.
import json
manifest_path = "data/splits_combined/split_manifest.json"
try:
    m = json.load(open(manifest_path))
    print(f"train = {m['n_train_annotated']} Annotated + {m['n_train_codipas']} CODIPAS "
          f"= {m['n_train_combined']} rows | val={m['n_val']} test={m['n_test']} (frozen Annotated benchmark)")
    print("\nper-class (combined train):")
    for cls, n in m["train_per_class_combined"].items():
        print(f"  {cls:22s} {n}")
except FileNotFoundError:
    print(f"[warn] {manifest_path} not found — did you push data/splits_combined/ to {BRANCH}?")

train = 2024 Annotated + 2621 CODIPAS = 4645 rows | val=253 test=253 (frozen Annotated benchmark)

per-class (combined train):
  no_distortion          1951
  emotional_reasoning    279
  overgeneralization     360
  mental_filter          179
  should_statements      274
  all_or_nothing         166
  mind_reading           369
  fortune_telling        273
  magnification          246
  personalization        190
  labeling               358


In [31]:
# 2. Choose ONE task, train it over three seeds for EACH model, evaluate each on val+test,
#    then aggregate everything at the end. --device auto resolves to the Kaggle GPU (cuda).
TASK   = "multilabel"  # one of: binary | multiclass | multilabel
SEEDS  = (42, 1337, 2024)
MODELS = [
    #"microsoft/deberta-v3-base",
    "mental/mental-roberta-base",   # already trained successfully — commented out for this run
]

# DeBERTa-v3's disentangled attention uses more memory per layer than RoBERTa's plain
# attention at the same batch size; it OOM'd at the default 16 on this GPU. Halve it here
# instead of touching the CLI default, so other models keep their normal batch size.
BATCH_SIZE = {
    "microsoft/deberta-v3-base": 8,
    "mental/mental-roberta-base": 16,
}

SPLITS_DIR = "data/splits_combined"

OUT_DIR = "results_combined"
!mkdir -p $OUT_DIR   # created up front so later cells never hit a missing-directory error

for model in MODELS:
    tag = model.split("/")[-1]
    bs = BATCH_SIZE.get(model, 16)
    print(f"\n=== {model} (batch_size={bs}) ===")
    for seed in SEEDS:
        ckpt = f"checkpoints/{TASK}_{tag}_{seed}"
        !python -m src.train_transformer --task $TASK --model $model --seed $seed --device auto --splits $SPLITS_DIR --batch-size $bs
        !python -m src.evaluate --checkpoint $ckpt --reference --splits $SPLITS_DIR --out $OUT_DIR

!python -m src.aggregate --results $OUT_DIR



=== mental/mental-roberta-base (batch_size=16) ===
[multilabel] device=cuda model=mental/mental-roberta-base train=4645 val=253 epochs=4
Loading weights: 100%|█| 197/197 [00:00<00:00, 3667.72it/s, Materializing param=
RobertaForSequenceClassification LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISS

In [24]:
# 4. Compare against the existing results/ (roberta-base, original Annotated_data splits)
#    and results_codipas/ (CODIPAS-only splits) already committed in the repo, alongside
#    this run's results_combined/ (DeBERTa-v3 this run + MentalRoBERTa from the prior run,
#    both on combined splits — both land in the same results_combined/paper_comparison.csv).
import pandas as pd
from pathlib import Path

SOURCES = {"results": "results", "results_codipas": "results_codipas", "results_combined": OUT_DIR}

frames = []
for label, folder in SOURCES.items():
    p = Path(folder) / "paper_comparison.csv"
    if p.exists():
        d = pd.read_csv(p)
        d["results_dir"] = label
        frames.append(d)
    else:
        print(f"[skip] {p} not found")

all_results = pd.concat(frames, ignore_index=True)
view = all_results[(all_results["task"] == TASK) & (all_results["split"] == "test")]
comparison = (
    view.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1"]]
    .agg(["mean", "std"]).round(3)
)
print(comparison)

all_results.to_csv(f"{OUT_DIR}/all_sources_comparison.csv", index=False)
print(f"\nWrote combined comparison table to {OUT_DIR}/all_sources_comparison.csv")

                                                 weighted_f1        macro_f1  \
                                                        mean    std     mean   
results_dir      model                                                         
results          mental/mental-bert-base-uncased       0.195  0.010    0.191   
                 roberta-base                          0.209  0.011    0.207   
results_codipas  mental/mental-bert-base-uncased       0.280  0.029    0.276   
results_combined mental/mental-roberta-base            0.283  0.017    0.277   
                 microsoft/deberta-v3-base             0.209  0.011    0.193   

                                                         
                                                    std  
results_dir      model                                   
results          mental/mental-bert-base-uncased  0.016  
                 roberta-base                     0.021  
results_codipas  mental/mental-bert-base-uncased  0.031  
results_com

In [25]:
# 5. Copy results_combined/ to the Kaggle output so it can be downloaded from the session.
!mkdir -p /kaggle/working/$OUT_DIR
!cp -r $OUT_DIR/* /kaggle/working/$OUT_DIR/
!ls -la /kaggle/working/$OUT_DIR

total 2032
drwxr-xr-x 2 root root   4096 Aug  6 13:15 .
drwxr-xr-x 5 root root   4096 Aug  6 13:15 ..
-rw-r--r-- 1 root root  15550 Aug  6 13:15 all_sources_comparison.csv
-rw-r--r-- 1 root root  36219 Aug  6 13:15 confusion_deberta-v3-base_binary_1337.png
-rw-r--r-- 1 root root  36324 Aug  6 13:15 confusion_deberta-v3-base_binary_2024.png
-rw-r--r-- 1 root root  36500 Aug  6 13:15 confusion_deberta-v3-base_binary_42.png
-rw-r--r-- 1 root root 126199 Aug  6 13:15 confusion_deberta-v3-base_multiclass_1337_no_nd.png
-rw-r--r-- 1 root root 139469 Aug  6 13:15 confusion_deberta-v3-base_multiclass_1337.png
-rw-r--r-- 1 root root 126639 Aug  6 13:15 confusion_deberta-v3-base_multiclass_2024_no_nd.png
-rw-r--r-- 1 root root 140155 Aug  6 13:15 confusion_deberta-v3-base_multiclass_2024.png
-rw-r--r-- 1 root root 125092 Aug  6 13:15 confusion_deberta-v3-base_multiclass_42_no_nd.png
-rw-r--r-- 1 root root 136379 Aug  6 13:15 confusion_deberta-v3-base_multiclass_42.png
-rw-r--r-- 1 root root  366

In [32]:
# Run Captum explainability on your newly fine-tuned MentalRoBERTa checkpoint
!python -m src.explain \
  --model "checkpoints/multilabel_mental-roberta-base_42" \
  --text "I really shouldn't be struggling with this code, I should just know how to do it by now."

Loading weights: 100%|█| 201/201 [00:00<00:00, 1593.44it/s, Materializing param=

--- Original Text ---
I really shouldn't be struggling with this code, I should just know how to do it by now.

--- Highlighted HTML Output ---
I really <mark style='background-color: #ffcccc; padding: 0.1em; border-radius: 3px; font-weight: bold;'>shouldn't</mark> be struggling with this code, I <mark style='background-color: #ffcccc; padding: 0.1em; border-radius: 3px; font-weight: bold;'>should</mark> just know how to do it by now.
